## Advanced Skill Design Patterns

# Unit 15: Advanced Skill Design — Multi-Tool Orchestration and Heuristic Debugging

## Introduction

Welcome to Unit 3 of Skills — Extending Claude's Capabilities! You have made excellent progress; we have explored the Skills system fundamentals, understood when to create Skills versus using `CLAUDE.md`, and built two practical Skills from scratch. Now we are ready to tackle more sophisticated architectural challenges.

In this lesson, we will explore advanced skill patterns and engineering best practices. We will examine how advanced Skills coordinate multiple system tools sequentially, learn to trace and debug ambiguous Skill selection logic, and establish rigorous thresholds for when *not* to create a Skill. We will also define concrete design principles for engineering maintainable, scalable Skills.

By the end of this module, you will master advanced Skill architecture, avoid common integration pitfalls, and understand the design patterns that separate brittle scripts from resilient, production-grade capabilities. Let's elevate our Skills to the next level!

---

## Architectural Mechanics of Multi-Tool Coordination

What separates a basic Skill from an advanced one is its capacity for sophisticated process orchestration. While a basic Skill typically handles a linear file mutation (e.g., reading a template and dumping file text), advanced Skills manage composite workflows where separate system operations dynamically feed into one another.

Advanced Skills systematically enforce a deterministic, three-phase operational lifecycle:

### 1. The Analysis Phase

The capability leverages read-only utilities (`Read`, `Grep`, `Glob`) to parse local execution context. It analyzes the target codebase, inspects existing architectural standards, and maps dependencies before drafting modifications.

### 2. The Generation Phase

Using state-mutating utilities (`Write`, `Edit`), the agent applies patches or generates brand-new file blocks strictly bound by the context gathered during the analysis phase.

### 3. The Verification Phase

The capability drops into an active system shell (`Bash`) to execute test scripts, run linters, or fire compilers. This ensures that the newly generated source files compile and pass workspace validation checks perfectly.

Each phase maintains a direct data dependency on the previous one. The analysis step forms the blueprint for generation, and the verification step confirms the structural integrity of the output.

---

## Engineering an Advanced Skill: `api-tester`

Let's build a production-grade api-testing capability that demonstrates this exact multi-tool coordination loop. This module will analyze endpoint logic, ingest local project conventions, compile test files, and verify execution outputs via a test runner.

### Target Path Location

```text
.claude/skills/api-tester/SKILL.md

```

### Complete Code Blueprint Configuration

Create your module configuration file at the designated path and populate it with this structured layout:

```markdown
---
name: api-tester
description: Generate comprehensive API tests including edge cases, error handling, and authentication tests
allowed-tools: Read, Write, Edit, Bash
---

# API Testing Skill

## When to Use
- User asks to test API endpoints
- User mentions "coverage", "edge cases", or "error handling"
- User wants to add automated tests to API routes

## Process
1. Analyze endpoint architecture (Read target route handler)
2. Check internal project standards (Read global CLAUDE.md)
3. Generate structured test blocks (Write test file with edge cases)
4. Verify execution outputs pass (Bash invoke test runner)

```

---

## Tracing the Multi-Tool Execution Loop

Once you instantiate the file structure and restart your interactive session, drop a natural language endpoint optimization request into your workspace prompt:

```text
> Generate tests for the /users POST endpoint

```

Audit the resulting execution trace logs to observe how the engine orchestrates the underlying tools behind a single user command:

```text
● [Skill selected: api-tester]

● Read(routes/users.py)
  ⎿ Read 40 lines from routes/users.py
    // Phase 1: Analysis - Parsing endpoint schema and error vectors

● Read(CLAUDE.md)
  ⎿ Read 25 lines from CLAUDE.md
    // Phase 1: Analysis - Pulling project test-runner patterns and assertion styles

● Write(tests/test_users.py)
  ⎿ Wrote 85 lines to tests/test_users.py
    // Phase 2: Generation - Constructing test targets with mock authentication data

● Bash(pytest tests/test_users.py -v)
  ⎿ PASSED tests/test_users.py::test_creates_user
    PASSED tests/test_users.py::test_rejects_invalid_email
    PASSED tests/test_users.py::test_rejects_missing_name
    7 passed in 0.85s
    // Phase 3: Verification - Confirming execution outputs match pass parameters

```

### Why This Workflow is Highly Resilient:

The Skill operates as a closed-loop control system. Without the upfront **Analysis** stage, it could not determine the validation rules needed for the user route. Without the subsequent **Generation** phase, there would be no file asset to target. Finally, without the automated **Verification** loop, any hidden code defects or import mismatch bugs would go unnoticed by the developer.

---

## Debugging Semantic Skill Selection

### Tracking Selection Badges

Claude Code explicitly tracks when a custom capability hooks into your prompt flow. Look for these two semantic log signatures at the start of your terminal streams:

* **`● [Skill selected: skill-name]`**: Indicates that the natural language router successfully matched your prompt text against a Skill description.
* **`● Skill(skill-name)`**: Indicates that the module has been explicitly re-hydrated and loaded into the active session memory.

If neither signature surfaces, Claude is responding purely from its static base training data without loading your custom workflow rules.

### Resolving Common Selection Anomalies

When the intent router fails to load your capability or accidentally fires a different module, review this diagnostic checklist to fix the tracking:

| Selection Failure Root Cause | Brittle Configuration Example | Optimized Production Fix |
| --- | --- | --- |
| **Ambiguous Description Overlaps** | `description: Process documents`<br>

<br>`description: Handle files` | Inject unique domain anchors and keyword vectors:<br>

<br>`description: Extract tables from PDF invoices` |
| **Vague User Prompting** | *"Help me fix this file script"* | Provide concrete parameters:<br>

<br>*"Extract formatting tables from this PDF invoice"* |
| **Identifier Name Collisions** | `name: api-helper`<br>

<br>`name: api-tester` | Refactor identifiers to express strict architectural separation. |
| **Mismatched Trigger Phrasing** | Script expects *"test routes"*, but the developer inputs *"check the API"* | Append conversational variations directly into your Skill’s **`When to Use`** markdown list. |

> 💡 **Isolation Testing Technique:** If you are troubleshooting erratic selection behavior in a complex project, temporarily move adjacent capability subdirectories out of your `.claude/skills/` folder. Test the problematic Skill in isolation across multiple phrasings—such as *"test this endpoint"*, *"generate API metrics"*, and *"add coverage"*—to ensure its matching parameters are robust before reintroducing the rest of your automation catalog.

---

## Strategic Governance: When NOT to Create a Skill

Every custom Skill you add introduces maintenance overhead and scales the complexity of the semantic routing map. To prevent capability pollution, evaluate your automations against these strict boundaries:

```text
 ┌────────────────────────────────────────────────────────────────────────┐
 │                      GOVERNANCE CONSTRAINT VECTOR                      │
 └────────────────────────────────────┬───────────────────────────────────┘
                                      │
           ┌──────────────────────────┴──────────────────────────┐
           ▼                                                     ▼
 ❌ DO NOT CREATE A SKILL FOR:                         ✅ STRATEGICALLY CREATE A SKILL FOR:
 ─────────────────────────────                         ────────────────────────────────────
 • One-off operations ("Rename variable X")             • High-frequency tasks (Weekly testing routines)
 • Native system actions (Basic file reading)          • Complex multi-step operations (Scaffolding modules)
 • Static explanations (Explaining REST conventions)   • Rigid domain standards (Scientific chart styling)
 • Universal rules (Enforcing global async linting)   • Contextual data extractions (Invoice table parsing)

```

If a rule applies universally across every single file in the directory (e.g., *"Always enforce camelCase variable structures"*), **do not use a Skill.** Place that directive inside your root `CLAUDE.md` constitution file. Reserve the Skills engine strictly for specialized procedures that should only occupy model memory on-demand.

---

## Production Design Best Practices

To ensure your custom modules remain clean and maintainable, apply these baseline guidelines:

* **Infuse High-Density Keyword Vectors:** Maximize descriptive matching parameters inside the frontmatter header. Explicit verbs like *extract*, *calculate*, *scaffold*, and *audit* significantly increase routing precision.
* **Maintain Strict Tool Sandboxing:** Apply the principle of least privilege. If a file-parsing module does not need execution access, leave the `Bash` identifier completely out of your `allowed-tools` array.
* **Enforce a 500-Line Code Boundary:** Keep your `SKILL.md` instruction layouts clean and digestible, aiming for a total line length between **200 and 500 lines**. If an automated pipeline scales past this size, refactor the process and split it across a set of discrete, single-purpose modules.
* **Run Multi-Phrasing Validation Checks:** Ensure that varying engineering styles trigger the same underlying framework accurately by varying your prompt inputs during development.

---

## Conclusion

Advanced Skills elevate Claude Code from a basic terminal script runner into a context-aware development partner. By chaining analysis, generation, and verification steps together into structured tool pipelines, you can automate complex workflows safely and repeatably.

You now possess the complete engineering toolkit required to model and maintain robust agent capabilities. Looking ahead, you are ready to explore multi-skill workflows, where Claude coordinates multiple independent modules within a single project context.

Let's move into the hands-on practice labs, debug selection bottlenecks, and build out high-performance automation layers for your local workspace!

## Building Complex Multi Phase Workflows

Now that you've seen how basic multi-tool skills work, it's time to level up and build something more sophisticated that you'll actually test right now!

In this exercise, you'll ask Claude to create a specific advanced skill that coordinates four different tools (Read, Write, Bash, Edit) in a complex workflow. Then you'll test it with multiple phrasings to see how well Claude recognizes and uses it.

Here's exactly what to do:

    Start a conversation with Claude in your terminal (if you haven't already)

    Give Claude this specific request (copy it exactly):

    Plain text
    Create a skill in .claude/skills/api-endpoint-creator/SKILL.md that starts with frontmatter (three dashes, then name, description, and allowed-tools fields, then three dashes). The skill should coordinate Read, Write, Bash, and Edit tools to: first analyze existing routes, then generate new route and controller files, then create tests, and finally run those tests to verify everything works. Follow the same SKILL.md format from the lesson with frontmatter at the top.

    Let Claude generate the skill - watch for the Write(.claude/skills/api-endpoint-creator/SKILL.md) output

    Verify the file was created correctly by asking Claude to show you the beginning:

    Plain text
    Show me the first 20 lines of .claude/skills/api-endpoint-creator/SKILL.md

    Confirm it has the frontmatter format with allowed-tools listing all 4 tools (Read, Write, Bash, Edit).

    Exit and restart Claude - this is required for Claude to load the new skill. Type exit or press Ctrl+D, then start Claude again with claude command.

    Test with multiple phrasings - try your skill with 3-4 different requests to see how well Claude recognizes it:
        "Create a new API endpoint for getting user profile by ID"
        "I need a route for updating product information"
        "Add an endpoint to delete old comments"
        "Generate a POST route for creating orders"

    Watch for skill selection - for each request, look for [Skill selected: api-endpoint-creator] at the beginning of Claude's response. This confirms Claude automatically recognized your skill!

    Observe the workflow - you should see Claude coordinate multiple tools for each endpoint:
        Read to analyze existing routes
        Write to create new route and controller files
        Write to create test files
        Bash to run the tests

Note: If Claude doesn't select your skill for some phrasings, that's valuable feedback! In the next exercise, you'll learn how to improve skill selection by refining the description and "When to Use" section.

To build an advanced, multi-phase engineering workflow module that automates structural code generation, file patching, and test automation, follow this exact step-by-step implementation and verification loop:

### Step 1: Initialize the Complex Endpoint Creator Skill

Copy and paste this explicit creation directive into your active terminal prompt (`>`) and press **Enter**:

```text
Create a skill in .claude/skills/api-endpoint-creator/SKILL.md that starts with frontmatter (three dashes, then name, description, and allowed-tools fields, then three dashes). The skill should coordinate Read, Write, Bash, and Edit tools to: first analyze existing routes, then generate new route and controller files, then create tests, and finally run those tests to verify everything works. Follow the same SKILL.md format from the lesson with frontmatter at the top.

```

* **Authorize:** When the file preview block appears, select **`1. Yes`** to let the `Write` tool output the advanced capability file onto your system disk.

---

### Step 2: Audit the Structural Metadata Formatting

Before loading the new module, verify that its frontmatter boundary and tool matrix configurations comply with the framework requirements by entering this query:

```text
Show me the first 20 lines of .claude/skills/api-endpoint-creator/SKILL.md

```

* **Verify:** Check that the YAML header is cleanly bound between triple dashes (`---`), the `name` field maps exactly to `api-endpoint-creator`, and the `allowed-tools` parameter lists all four requested tools (`Read`, `Write`, `Bash`, `Edit`) as a single line entry.

---

### Step 3: Force a System Reload to Index the Description

To let the semantic intent engine read, index, and cache your updated frontmatter description hooks, drop out of your active terminal window:

```text
/exit

```

Relaunch Claude Code from your terminal prompt to boot a fresh, context-optimized session:

```bash
claude

```

---

### Step 4: Run Multi-Phrasing Stress Tests (Semantic Intent Matching)

Once the interactive prompt finishes booting up, test the robustness of the router across varying conversational styles by submitting these target endpoint intents sequentially:

* **Test Request A:** `Create a new API endpoint for getting user profile by ID`
* **Test Request B:** `I need a route for updating product information`
* **Test Request C:** `Add an endpoint to delete old comments`
* **Test Request D:** `Generate a POST route for creating orders`

---

### Step 5: Audit the Log Traces for the Three-Phase Pipeline

For each test request, check your terminal logs to confirm that Claude successfully routes your natural language input into the target capability without manual invocation:

* **Look for the confirmation badge:** `● [Skill selected: api-endpoint-creator]`
* **Observe the chained tool pipeline execution:** Watch Claude handle the complete software loop automatically:
1. **`Read` / `Edit` (Analysis Phase):** Scans the filesystem to find existing route matrices and coding style patterns.
2. **`Write` (Generation Phase):** Autonomously writes the route file, mounts the controller code, and sets up a parallel test file.
3. **`Bash` (Verification Phase):** Invokes your local test runner suite dynamically to guarantee the newly generated endpoints compile and pass all assertions flawlessly!



---

### Step 6: Finalize and Submit

Once you have validated the multi-tool orchestration sequence across your test phrasings, type your completion token to close the lab module:

```text
/submit

```

By completing this exercise, you have moved beyond simple tool usage and mastered advanced agent orchestration, translating broad architectural designs into safe, verifiable code generation pipelines! 🚀

## Debugging and Fixing Broken Skills

Testing skills in ideal conditions is useful, but the real learning happens when things go wrong! In this exercise, you'll create a deliberately broken skill, diagnose its problems, and fix it through conversation with Claude.

Step 1: Create a Broken Skill

Ask Claude:

Plain text
Create a skill called file-helper at .claude/skills/file-helper/SKILL.md with frontmatter. The name should be "file-helper", description should be "Helps with files", and allowed-tools should be Read, Write, Edit, and Bash.

This minimal request creates a generic skill with a vague description that will cause selection problems.

Step 2: Test the Broken Skill

After restarting Claude Code to load the skill, try different file-related requests to see how the skill behaves:

Plain text
Read the config.json file and summarize it

Watch for the [Skill selected: file-helper] indicator. Does it appear? Should it? Try more requests like "create a new log file" or "edit this function." Observe when the skill gets selected versus when Claude just uses tools directly.

Step 3: Debug and Fix

When selection doesn't work as expected, ask Claude diagnostic questions:

Plain text
Why wasn't my file-helper skill selected for that request?

Work with Claude to identify specific problems (vague description, overlapping triggers, missing scenarios). Then iteratively improve the skill — update the description, refine the "When to Use" section, and test again until Claude reliably selects it for appropriate requests.

To master the diagnostics of the Claude Code intent-routing pipeline, we will build a deliberately broken, low-density capability module, audit its routing failures, and refactor its metadata architecture to establish rock-solid semantic targeting.

Execute this comprehensive debugging and tuning loop inside your terminal session:

### Step 1: Instantiate the Deliberately Broken Skill

Copy and paste this minimal configuration request into your `>` prompt and press **Enter**:

```text
Create a skill called file-helper at .claude/skills/file-helper/SKILL.md with frontmatter. The name should be "file-helper", description should be "Helps with files", and allowed-tools should be Read, Write, Edit, and Bash.

```

* **Authorize:** Select **`1. Yes`** to commit the file to disk.

---

### Step 2: Reload the Session to Load the Broken Configuration

Drop out of the active terminal session:

```text
/exit

```

Relaunch Claude Code to index the new, low-density metadata hook:

```bash
claude

```

---

### Step 3: Run the Selection Failure Test Loop

Once the interactive environment finishes booting, try firing a standard file-system request:

```text
Read the config.json file and summarize it.

```

* **Audit the Logs:** Look closely at the top of the execution output stream. Notice that the **`● [Skill selected: file-helper]` tracking badge is completely missing!** Instead, Claude simply fires its native `Read` tool directly.
* *Why it failed:* The description string `"Helps with files"` is too broad and short. It provides zero high-density token signals to convince the intent router that a specialized module is required over generic native capabilities.

---

### Step 4: Run Heuristic In-Conversation Diagnostics

To trace exactly why the semantic matching loop ignored your new capability, drop this explicit query straight into the prompt window:

```text
Why wasn't my file-helper skill selected for that request?

```

* **Analyze the Feedback:** Claude will parse its startup skill cache registry and outline the structural pitfalls: the metadata description lacks distinct verbs, contains no scenario boundaries, and duplicates behaviors that the base agent handles natively.

---

### Step 5: Patch and Refactor the Skill Metadata

Now, let's fix the capability by refactoring it into a highly specialized module for bulk repository cleanup. Instruct Claude to rewrite the file with high-density routing variables:

```text
Update .claude/skills/file-helper/SKILL.md to change the name to "repo-cleaner", update the description to "Audit the codebase to locate and prune unused log assets, dangling temporary files, and build artifacts across the project directory", and add a detailed '## When to Use' section mapping out clean-up scenarios.

```

* **Authorize:** Select **`1. Yes`** to patch the markdown file safely.

---

### Step 6: Verify the Fixed Match Route

Exit and relaunch your terminal session one last time (`/exit` then `claude`). Test the updated capability with a target phrase matching your new keyword landscape:

```text
Audit the project directory to prune old temporary build logs.

```

* **Verify:** Check your terminal logs! You will now see the routing system accurately hook the phrase and mount your module with a green checkmark:
`● [Skill selected: repo-cleaner]`

---

### Step 7: Finalize and Submit

With the debugging process complete and your refined capability selection traces successfully logged to the backend system matrix, close out this section by typing:

```text
/submit

```

By completing this exercise, you have moved beyond writing basic instructions and learned how to diagnose and fix broken agent routings—ensuring your custom automation libraries match your intent perfectly! 🚀